In [2]:
# ============================================================
# NOTEBOOK 2: CLEAN DATA
# Purpose: Clean raw data, encode labels, save clean_dataset.csv
# ============================================================

import warnings
from pathlib import Path

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# Raw Excel file location
RAW_DATA_FILE = Path(
    r"D:\iit\DSGP\NutriScanner\health-risk-recommendations\data\health_nutrition_disease_dataset_12000.xlsx"
)

# Notebook working folder
BASE_DIR = Path(
    r"D:\iit\DSGP\NutriScanner\health-risk-recommendations\research\NoteBooks"
)

# Output folder for cleaned data
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

DATA_FILE = RAW_DATA_FILE
CLEAN_FILE = DATA_DIR / "clean_dataset.csv"

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Dataset not found:\n{DATA_FILE}")

DISEASE_COLUMNS = [
    "Diabetes_Risk",
    "Hypertension_Risk",
    "Heart_Disease_Risk",
    "Obesity_Risk",
    "Anemia_Risk",
    "Kidney_Disease_Risk"
]

FEATURES = [
    "Age", "Gender", "BMI",
    "Daily_Calories_kcal",
    "Carbohydrates_g",
    "Protein_g",
    "Total_Fat_g",
    "Saturated_Fat_g",
    "Trans_Fat_g",
    "Total_Sugar_g",
    "Added_Sugar_g",
    "Fiber_g",
    "Sodium_mg",
    "Potassium_mg",
    "Calcium_mg",
    "Iron_mg",
    "Vitamin_D_IU",
    "Vitamin_B12_mcg",
    "Physical_Activity_min",
    "Water_Intake_L"
]

USE_COLUMNS = FEATURES + DISEASE_COLUMNS

print("Raw dataset file:", DATA_FILE)
print("Raw file exists:", DATA_FILE.exists())
print("Clean file will be saved to:", CLEAN_FILE)

Raw dataset file: D:\iit\DSGP\NutriScanner\health-risk-recommendations\data\health_nutrition_disease_dataset_12000.xlsx
Raw file exists: True
Clean file will be saved to: D:\iit\DSGP\NutriScanner\health-risk-recommendations\research\NoteBooks\data\clean_dataset.csv


In [3]:
df = pd.read_excel(DATA_FILE, usecols=USE_COLUMNS)

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully
Shape: (12000, 26)


,Age,Gender,BMI,Daily_Calories_kcal,Carbohydrates_g,Protein_g,Total_Fat_g,Saturated_Fat_g,Trans_Fat_g,Total_Sugar_g,...,Vitamin_D_IU,Vitamin_B12_mcg,Physical_Activity_min,Water_Intake_L,Diabetes_Risk,Hypertension_Risk,Heart_Disease_Risk,Obesity_Risk,Anemia_Risk,Kidney_Disease_Risk
0,65,Female,31.9,2723,256,90,58,68,3.74,85,...,861,4.21,78,2.64,High,High,Low,Low,Low,Low
1,22,Male,40.7,2315,311,131,138,24,4.43,79,...,348,1.32,77,1.04,High,Low,Low,Low,Low,High
2,43,Female,20.8,2519,206,201,166,20,2.67,206,...,199,2.16,163,2.17,Low,Low,Low,Low,Low,Low
3,72,Female,19.6,3536,228,71,52,48,2.79,5,...,1001,3.61,191,3.85,Low,Low,Low,Low,Low,Low
4,21,Female,26.2,2285,257,96,45,57,4.17,54,...,19,2.66,89,3.48,Low,Low,Low,Low,Low,Low


In [4]:
# --------------------  Basic cleaning + robust encoding --------------------

def clean_text(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    text = text.replace("_", " ").replace("-", " ")
    text = " ".join(text.split())
    return text

def encode_gender(value):
    if pd.isna(value):
        return np.nan

    # handle numeric gender already encoded
    if isinstance(value, (int, float, np.integer, np.floating)):
        if float(value) == 0.0:
            return 0
        if float(value) == 1.0:
            return 1

    text = clean_text(value)

    if text in ["male", "m", "0", "0.0"]:
        return 0
    if text in ["female", "f", "1", "1.0"]:
        return 1

    return np.nan

def encode_risk(value):
    if pd.isna(value):
        return np.nan

    # handle numeric target already encoded
    if isinstance(value, (int, float, np.integer, np.floating)):
        if float(value) == 0.0:
            return 0
        if float(value) == 1.0:
            return 1

    text = clean_text(value)

    if text in ["low", "low risk", "0", "0.0", "false", "no"]:
        return 0
    if text in ["high", "high risk", "1", "1.0", "true", "yes", "at risk"]:
        return 1

    return np.nan

# remove exact duplicate rows
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print("Duplicates removed:", before - after)

print("\nRaw Gender values:")
print(df["Gender"].astype(str).value_counts(dropna=False).head(10))

for disease in DISEASE_COLUMNS:
    print(f"\nRaw values in {disease}:")
    print(df[disease].astype(str).value_counts(dropna=False).head(10))

# encode gender
df["Gender"] = df["Gender"].apply(encode_gender)

# encode disease targets
for disease in DISEASE_COLUMNS:
    df[disease] = df[disease].apply(encode_risk)

print("\nAfter encoding:")
print("Gender counts:")
print(df["Gender"].value_counts(dropna=False))

for disease in DISEASE_COLUMNS:
    print(f"\n{disease} counts:")
    print(df[disease].value_counts(dropna=False))

df.head()

Duplicates removed: 0

Raw Gender values:
Gender
Female    6031
Male      5969
Name: count, dtype: int64

Raw values in Diabetes_Risk:
Diabetes_Risk
Low     8575
High    3425
Name: count, dtype: int64

Raw values in Hypertension_Risk:
Hypertension_Risk
Low     9218
High    2782
Name: count, dtype: int64

Raw values in Heart_Disease_Risk:
Heart_Disease_Risk
Low     10933
High     1067
Name: count, dtype: int64

Raw values in Obesity_Risk:
Obesity_Risk
Low     10157
High     1843
Name: count, dtype: int64

Raw values in Anemia_Risk:
Anemia_Risk
Low     11380
High      620
Name: count, dtype: int64

Raw values in Kidney_Disease_Risk:
Kidney_Disease_Risk
Low     10689
High     1311
Name: count, dtype: int64

After encoding:
Gender counts:
Gender
1    6031
0    5969
Name: count, dtype: int64

Diabetes_Risk counts:
Diabetes_Risk
0    8575
1    3425
Name: count, dtype: int64

Hypertension_Risk counts:
Hypertension_Risk
0    9218
1    2782
Name: count, dtype: int64

Heart_Disease_Risk counts:


,Age,Gender,BMI,Daily_Calories_kcal,Carbohydrates_g,Protein_g,Total_Fat_g,Saturated_Fat_g,Trans_Fat_g,Total_Sugar_g,...,Vitamin_D_IU,Vitamin_B12_mcg,Physical_Activity_min,Water_Intake_L,Diabetes_Risk,Hypertension_Risk,Heart_Disease_Risk,Obesity_Risk,Anemia_Risk,Kidney_Disease_Risk
0,65,1,31.9,2723,256,90,58,68,3.74,85,...,861,4.21,78,2.64,1,1,0,0,0,0
1,22,0,40.7,2315,311,131,138,24,4.43,79,...,348,1.32,77,1.04,1,0,0,0,0,1
2,43,1,20.8,2519,206,201,166,20,2.67,206,...,199,2.16,163,2.17,0,0,0,0,0,0
3,72,1,19.6,3536,228,71,52,48,2.79,5,...,1001,3.61,191,3.85,0,0,0,0,0,0
4,21,1,26.2,2285,257,96,45,57,4.17,54,...,19,2.66,89,3.48,0,0,0,0,0,0


In [7]:
# --------------------  Convert numeric columns + handle missing --------------------

numeric_cols = [col for col in FEATURES if col != "Gender"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Missing target values before dropping:")
print(df[DISEASE_COLUMNS].isnull().sum())

before = len(df)
df = df.dropna(subset=DISEASE_COLUMNS).reset_index(drop=True)
after = len(df)

print("Rows dropped because of missing target values:", before - after)
print("Shape after dropping missing targets:", df.shape)

for col in FEATURES:
    if df[col].isnull().sum() > 0:
        if col == "Gender":
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

print("\nRemaining missing values:")
print(df.isnull().sum())
print("\nTotal remaining missing values:", int(df.isnull().sum().sum()))

Missing target values before dropping:
Diabetes_Risk          0
Hypertension_Risk      0
Heart_Disease_Risk     0
Obesity_Risk           0
Anemia_Risk            0
Kidney_Disease_Risk    0
dtype: int64
Rows dropped because of missing target values: 0
Shape after dropping missing targets: (12000, 26)

Remaining missing values:
Age                      0
Gender                   0
BMI                      0
Daily_Calories_kcal      0
Carbohydrates_g          0
Protein_g                0
Total_Fat_g              0
Saturated_Fat_g          0
Trans_Fat_g              0
Total_Sugar_g            0
Added_Sugar_g            0
Fiber_g                  0
Sodium_mg                0
Potassium_mg             0
Calcium_mg               0
Iron_mg                  0
Vitamin_D_IU             0
Vitamin_B12_mcg          0
Physical_Activity_min    0
Water_Intake_L           0
Diabetes_Risk            0
Hypertension_Risk        0
Heart_Disease_Risk       0
Obesity_Risk             0
Anemia_Risk             

In [8]:
# --------------------  Remove duplicate feature rows --------------------

before = len(df)
df = df.drop_duplicates(subset=FEATURES).reset_index(drop=True)
after = len(df)

print("Duplicate feature rows removed:", before - after)
print("Final shape after duplicate-feature removal:", df.shape)

df.head()

Duplicate feature rows removed: 0
Final shape after duplicate-feature removal: (12000, 26)


,Age,Gender,BMI,Daily_Calories_kcal,Carbohydrates_g,Protein_g,Total_Fat_g,Saturated_Fat_g,Trans_Fat_g,Total_Sugar_g,...,Vitamin_D_IU,Vitamin_B12_mcg,Physical_Activity_min,Water_Intake_L,Diabetes_Risk,Hypertension_Risk,Heart_Disease_Risk,Obesity_Risk,Anemia_Risk,Kidney_Disease_Risk
0,65,1,31.9,2723,256,90,58,68,3.74,85,...,861,4.21,78,2.64,1,1,0,0,0,0
1,22,0,40.7,2315,311,131,138,24,4.43,79,...,348,1.32,77,1.04,1,0,0,0,0,1
2,43,1,20.8,2519,206,201,166,20,2.67,206,...,199,2.16,163,2.17,0,0,0,0,0,0
3,72,1,19.6,3536,228,71,52,48,2.79,5,...,1001,3.61,191,3.85,0,0,0,0,0,0
4,21,1,26.2,2285,257,96,45,57,4.17,54,...,19,2.66,89,3.48,0,0,0,0,0,0


In [9]:
# --------------------  Save clean dataset --------------------

print("Final dataset shape before saving:", df.shape)

if df.empty:
    raise ValueError("DataFrame is empty. Check the raw target values printed in Cell 3.")

df.to_csv(CLEAN_FILE, index=False)
print("Clean dataset saved to:", CLEAN_FILE)

Final dataset shape before saving: (12000, 26)
Clean dataset saved to: D:\iit\DSGP\NutriScanner\health-risk-recommendations\research\NoteBooks\data\clean_dataset.csv
